In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
SOURCE = "/Volumes/databricks_assesment/landing/landing_volume/trade_events/"
BRONZE = "databricks_assesment.bronze.stream_bronze_trade_events"
SILVER = "databricks_assesment.silver.stream_silver_trades"
GOLD = "databricks_assesment.gold.stream_gold_symbol_metrics"
CHECKPOINT = "/Volumes/databricks_assesment/landing/landing_volume/checkpoints/trades"

In [0]:
schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_ts", StringType()),
    StructField("trade_id", StringType()),
    StructField("account_id", StringType()),
    StructField("symbol", StringType()),
    StructField("side", StringType()),
    StructField("quantity", DoubleType()),
    StructField("price", DoubleType()),
    StructField("currency", StringType()),
    StructField("event_type", StringType())
])

In [0]:
# Auto Loader = cloudFiles + Structured Streaming
raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(schema)
        .load(SOURCE)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_name"))
)

In [0]:
q1 = (raw.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", CHECKPOINT + "/bronze")
      .trigger(availableNow=True)
      .toTable(BRONZE))

In [0]:
%sql
SELECT COUNT(*) FROM databricks_assesment.bronze.stream_bronze_trade_events

In [0]:
%sql
SELECT * FROM databricks_assesment.bronze.stream_bronze_trade_events

In [0]:
bronze_stream = spark.readStream.table(BRONZE)

In [0]:
%sql
SELECT
    event_id,
    COUNT(*) AS duplicate_count
FROM databricks_assesment.bronze.stream_bronze_trade_events
GROUP BY event_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) FROM databricks_assesment.silver.stream_silver_trades

In [0]:
silver = (
    bronze_stream
        .withColumn("event_time", F.to_timestamp("event_ts"))
        .withColumn("trade_value", F.col("quantity") * F.col("price"))
        .withWatermark("event_time", "10 minutes")
        .dropDuplicates(["event_id"])
)

In [0]:
q2 = (silver.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", CHECKPOINT + "/silver")
      .trigger(availableNow=True)
      .toTable(SILVER))

In [0]:
%sql
SELECT COUNT(*) FROM databricks_assesment.silver.stream_silver_trades

In [0]:
%sql
SELECT * FROM databricks_assesment.silver.stream_silver_trades

In [0]:
%sql
DESCRIBE HISTORY databricks_assesment.silver.stream_silver_trades

In [0]:
gold = (
    spark.readStream
        .table(SILVER)
        .withWatermark("event_time", "10 minutes")
        .groupBy(
            F.window("event_time", "5 minutes"),
            "symbol"
        )
        .agg(
            F.count("*").alias("trade_count"),
            F.sum("trade_value").alias("gross_value"),
            F.sum(
                F.when(F.col("side") == "BUY", F.col("trade_value"))
                 .otherwise(0)
            ).alias("buy_value"),
            F.sum(
                F.when(F.col("side") == "SELL", F.col("trade_value"))
                 .otherwise(0)
            ).alias("sell_value")
        )
)

In [0]:
q3 = (
    gold.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT + "/gold_v2")
        .trigger(availableNow=True)
        .toTable(GOLD)
)

In [0]:
%sql
DESCRIBE HISTORY databricks_assesment.gold.stream_gold_symbol_metrics

In [0]:
%sql
SELECT COUNT(*)
FROM databricks_assesment.gold.stream_gold_symbol_metrics;

In [0]:
%sql
SELECT * FROM databricks_assesment.gold.stream_gold_symbol_metrics

In [0]:
%sql
SELECT COUNT(*)
FROM databricks_assesment.gold.stream_gold_symbol_metrics

In [0]:
%sql
SELECT MAX(window.start),MAX(window.end) FROM databricks_assesment.gold.stream_gold_symbol_metrics